# 01 — Full-Text, Vector, and Hybrid Search on the Same Query

A demonstration, not a build-along: it queries the finished index you build in Module 2.


## Setup — run once per kernel session

In [2]:
# Connects to the index Module 2 built. Run once per kernel session.
import re
import sys
sys.path.insert(0, "..")

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

from config import load_config

config = load_config()
credential = DefaultAzureCredential()
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

print("Querying index:", config.search_index_name)


Querying index: state-driver-manuals


In [3]:
# Helpers reused by the cells below.
def top_hits(query, state_filter=None, k=3):
    """Hybrid + semantic search: the top k passages."""
    return list(search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")],
        query_type="semantic",
        semantic_configuration_name="semantic-config",
        filter=f"state eq '{state_filter}'" if state_filter else None,
        top=k,
        select=["state", "title", "chunk"],
    ))


GARBLED_CHAR = "\ufffd"  # U+FFFD REPLACEMENT CHARACTER — PDF-encoding noise, not content


def show_results(hits, label="", answer_pattern=r"\d+\s*mph", preview_len=220):
    """Print hits, flagging garbled text and passages that match answer_pattern."""
    if not hits:
        print(f"{label}: no results")
        return
    print(label)
    for hit in hits:
        full = hit["chunk"].strip().replace("\n", " ")
        preview = full[:preview_len]
        reranker = hit.get("@search.reranker_score")
        if preview.count(GARBLED_CHAR) > 3:
            flag = "  <-- GARBLED, do not read"
        elif re.search(answer_pattern, preview, re.IGNORECASE):
            flag = "  <-- READ THIS ONE"
        else:
            flag = ""
        print(f"  [{hit['state']} — {hit['title']}, reranker={reranker:.2f}]{flag}")
        print(f"    {preview}...")
    print()


def mode_table(query, key_pattern, label, state_filter=None, k=3, preview_len=90):
    """Print the top k passages per search mode; "yes" marks a match for the regex key_pattern."""
    flt = f"state eq '{state_filter}'" if state_filter else None
    vector_query = [VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")]
    select = ["state", "chunk"]
    modes = {
        "keyword": search_client.search(search_text=query, filter=flt, top=k, select=select),
        "vector": search_client.search(search_text=None, vector_queries=vector_query, filter=flt, top=k, select=select),
        "hybrid": search_client.search(search_text=query, vector_queries=vector_query, filter=flt, top=k, select=select),
    }
    summary = []
    for mode, results in modes.items():
        hits = list(results)
        matched = 0
        first_correct = None
        print(mode.upper())
        for rank, hit in enumerate(hits, 1):
            text = " ".join(hit["chunk"].replace(GARBLED_CHAR, "").split())
            found = re.search(key_pattern, text, re.IGNORECASE)
            matched += bool(found)
            if found and first_correct is None:
                first_correct = rank
            start = max(0, found.start() - 30) if found else 0
            if start:
                start = text.find(" ", start) + 1 or start  # begin at a word boundary
            mark = "yes" if found else "no "
            print(f"  {rank}. {hit['state']:<11} {mark}  ...{text[start:start + preview_len]}...")
        first = f"#{first_correct}" if first_correct else "none"
        summary.append(f"{mode} {matched}/{len(hits)} (first correct: {first})")
        print()
    print(f"Passages containing \"{label}\":")
    for line in summary:
        print("  " + line)


---
## Clip 1 — The Retrieval Layer and Why Enterprises Need One

Two runs of the same question against the finished index; only the filter changes.


In [4]:
hero_question = "What's the speed limit in a school zone?"

unfiltered = top_hits(hero_question)
show_results(unfiltered, "Run 1 — unfiltered")


Run 1 — unfiltered
  [wisconsin — wisconsin-motorists-handbook.pdf, reranker=3.01]
    rain, snow and/or ice.  Speed limits:  Speed limits are the fastest speed you are allowed to drive in the area you are in. These signs are on  the side of most roads. It is illegal to go over the speed limit that is on t...
  [florida — florida-driver-handbook.pdf, reranker=3.01]  <-- GARBLED, do not read
    less you can see.        68 	 ©2023 Florida Department of Highway Safety and Motor Vehicles  	���������������������������������������������������������������������������������������������������������������  Right-of-Way ...
  [florida — florida-driver-handbook.pdf, reranker=2.97]  <-- READ THIS ONE
    the speed limit and may  receive a citation for speeding.   Florida Standard Speed Limits 	◆ School Zones: 20 MPH  	◆ Municipal, Business, or Residential  Area: 30 MPH*  	◆ Streets and Highways: 55 MPH*  	◆ Limited Acces...



In [5]:
filtered = top_hits(hero_question, state_filter="florida")
show_results(filtered, "Run 2 — filter='state eq \'florida\''")

Run 2 — filter='state eq 'florida''
  [florida — florida-driver-handbook.pdf, reranker=3.01]  <-- GARBLED, do not read
    less you can see.        68 	 ©2023 Florida Department of Highway Safety and Motor Vehicles  	���������������������������������������������������������������������������������������������������������������  Right-of-Way ...
  [florida — florida-driver-handbook.pdf, reranker=2.97]  <-- READ THIS ONE
    the speed limit and may  receive a citation for speeding.   Florida Standard Speed Limits 	◆ School Zones: 20 MPH  	◆ Municipal, Business, or Residential  Area: 30 MPH*  	◆ Streets and Highways: 55 MPH*  	◆ Limited Acces...
  [florida — florida-driver-handbook.pdf, reranker=2.69]
    highlight the danger zone where road  users should not stop. These markings indicate the clearance needed  for trains to safely pass, as any object within the dynamic envelope  has the potential to be struck when the tra...



---
## Clip 2 — Full-Text, Vector, and Hybrid Search on the Same Query

The same questions under keyword (`search_text=`), vector (`vector_queries=`), and hybrid (both) search.


In [9]:
# TLSAE = Traffic Law and Substance Abuse Education, a Florida course.
# It appears in only one of the five manuals.
print("Query A — exact acronym:")
mode_table("What is TLSAE?", key_pattern=r"TLSAE", label="TLSAE")

Query A — exact acronym:
KEYWORD
  1. florida     yes  ...Substance Abuse Education (TLSAE) Course If you have never held—or do not have in your pos...
  2. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  3. florida     yes  ...Substance Abuse Education (TLSAE) Course Basic Driver Improvement (BDI) Course Advanced Dr...

VECTOR
  1. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  2. florida     yes  ...school are exempt from the TLSAE requirement Parental Consent for Minors If you are under ...
  3. california  no   ...are treated fairly, consistently with laws and regulations, and with dignity and respect. ...

HYBRID
  1. florida     yes  ...Abuse Education Course (TLSAE)*. 3. Pass Vision and Class E Knowledge exams. Driver Licens...
  2. florida     yes  ...school are exempt from the TLSAE requirement Parental Consent for Minors If you are under ...

In [10]:
# The manuals say "following distance" or "four-second rule", never "space in front of me".
print("Query B — paraphrase, the manuals use different words:")
mode_table(
    "How much space should I leave in front of me?",
    key_pattern=r"following distance|second rule",
    label="following distance / second rule",
)

Query B — paraphrase, the manuals use different words:
KEYWORD
  1. virginia    no   ...you are backing up, have someone outside to help guide you. To back the trailer to the lef...
  2. florida     no   ...unless it is a path or part of a roadway reserved for bicycles. They must ride single file...
  3. california  no   ...whether you have enough space to pass whenever you approach: • An oncoming vehicle or bicy...

VECTOR
  1. wisconsin   yes  ...front: Use the “four second following distance rule” as explained previously when you are ...
  2. virginia    yes  ...the two-, three- and four-second rule to determine if you are following far enough behind ...
  3. virginia    yes  ...the two-, three- and four-second rule to determine if...

HYBRID
  1. wisconsin   yes  ...front: Use the “four second following distance rule” as explained previously when you are ...
  2. virginia    yes  ...the two-, three- and four-second rule to determine if you are following far enough behind ...
  3. v

In [11]:
print("Query C — a plain question:")
mode_table(
    "What does a flashing yellow arrow mean?",
    key_pattern=r"flashing yellow arrow",
    label="flashing yellow arrow",
)

Query C — a plain question:
KEYWORD
  1. florida     no   ...charged with the safety belt violation? 15. What does a single broken white line on the hi...
  2. florida     no   ...OF ATTORNEY GENERAL ATTORNE ASHLEY MOODY STATE OF FLORIDA Stronger, Safer Florida https://...
  3. virginia    no   ...to vehicles coming from the other direction and pedestrians in the intersection. Be sure t...

VECTOR
  1. virginia    no   ...if a red arrow pointing to the right is displayed. Left turn on red: You may turn left at ...
  2. virginia    yes  ...hazardous conditions. Flashing yellow arrow: At a flashing yellow arrow, you may turn in t...
  3. california  yes  ...You do not need to stop. Flashing Yellow Arrow You can turn, but your turn is not protecte...

HYBRID
  1. virginia    yes  ...hazardous conditions. Flashing yellow arrow: At a flashing yellow arrow, you may turn in t...
  2. california  yes  ...You do not need to stop. Flashing Yellow Arrow You can turn, but your turn is not protecte